# Regression detection for OTel GenAI traces — Kalibra

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/khan5v/kalibra/blob/main/examples/otel_genai/otel_genai_tutorial.ipynb)

If your agent traces have `gen_ai.*` attributes — from the [OTel GenAI semantic conventions](https://opentelemetry.io/docs/specs/semconv/gen-ai/) — this notebook shows how to catch regressions that aggregate metrics hide. Validated with `opentelemetry-instrumentation-openai-v2`, compatible with any exporter that preserves standard `gen_ai.*` span attributes.

**No API key needed.** Pre-recorded traces from a real OpenAI gpt-4o-mini agent are included. All analysis cells run as-is.

**The experiment:** Same agent, same 25 tasks. Baseline uses `max_tokens=2048`. Current uses `max_tokens=150`. Aggregate token usage drops 62% — but the confidence interval is so wide that the change isn't even statistically significant. Meanwhile, 14 out of 25 task types silently truncate — and the per-task breakdown catches every one of them.

**Stack:** [OpenTelemetry GenAI](https://opentelemetry.io/docs/specs/semconv/gen-ai/) (trace format) → [Kalibra](https://github.com/khan5v/kalibra) (statistical comparison + CI gates)

In [ ]:
!pip install -q "kalibra>=0.2.2"

## What OTel GenAI traces look like

Each trace has 2 spans: a root "agent" span (your application code) wrapping a "chat gpt-4o-mini" span (auto-instrumented by `opentelemetry-instrumentation-openai-v2`).

```
agent                              <- your span (variant, task.id in attributes)
 └── chat gpt-4o-mini              <- auto-instrumented (gen_ai.* attributes)
       gen_ai.usage.input_tokens: 42
       gen_ai.usage.output_tokens: 150
       gen_ai.response.finish_reasons: ["length"]    <- truncated!
       gen_ai.request.model: "gpt-4o-mini"
       gen_ai.system: "openai"
```

**What's different from OpenInference:** No `llm.cost.total` — the OTel GenAI spec has no cost attribute. No `openinference.span.kind` — span classification uses `gen_ai.operation.name` instead. Finish reasons are a top-level array attribute, not buried in serialized JSON. Kalibra auto-detects this format and maps `["length"]` to outcome = failure.

In [ ]:
import os, json
from pathlib import Path
from kalibra.loader import load_traces
from kalibra.config import parse_matcher

# ── Resolve trace file ────────────────────────────────────────────────
candidates = [
    Path("traces.jsonl"),                          # same directory as notebook
    Path("examples/otel_genai/traces.jsonl"),      # from repo root
]
TRACES_PATH = next((str(p) for p in candidates if p.exists()), None)

if TRACES_PATH is None:
    import urllib.request
    os.makedirs("traces", exist_ok=True)
    url = "https://raw.githubusercontent.com/khan5v/kalibra/main/examples/otel_genai/traces.jsonl"
    TRACES_PATH = "traces/traces.jsonl"
    print("Downloading pre-recorded traces...")
    urllib.request.urlretrieve(url, TRACES_PATH)

# ── Load and split ────────────────────────────────────────────────────
all_traces = load_traces(TRACES_PATH)

m_base = parse_matcher("variant == baseline")
m_curr = parse_matcher("variant == current")
baseline = [t for t in all_traces if m_base.matches(t.metadata.get(m_base.field))]
current = [t for t in all_traces if m_curr.matches(t.metadata.get(m_curr.field))]

print(f"Loaded {len(all_traces)} traces from {TRACES_PATH}")
print(f"  Baseline: {len(baseline)} traces (variant == baseline)")
print(f"  Current:  {len(current)} traces (variant == current)")

In [ ]:
from kalibra.engine import compare
from kalibra.renderers import render
from IPython.display import Markdown

result = compare(
    baseline, current,
    require=[
        "token_delta_pct <= -10",  # tokens decreased — but CI is wide, gate SKIPPED
        "regressions <= 2",         # per-task regressions — gate FAILS (14 tasks broke)
    ],
    baseline_source="max_tokens=2048",
    current_source="max_tokens=150",
    metric_config={"trace_breakdown": {"task_id_field": "task.id"}},
)

Markdown(render(result, "markdown", verbose=True))

## Reading the results

Three things to notice:

**1. Token usage dropped — but the gate was skipped.** Median tokens per trace fell ~62%, but the confidence interval is wide (roughly -81% to +49%). Kalibra skips the `token_delta_pct` gate when the change isn't statistically significant. The aggregate number *looks* like an improvement but can't be trusted.

**2. Success rate collapsed.** 100% to 44%. The `gen_ai.response.finish_reasons` attribute tells the story — complex tasks hit the 150-token limit and returned `["length"]` instead of `["stop"]`. Kalibra maps `length` to failure automatically for OTel GenAI traces.

**3. Cost shows "no cost data."** This is correct — the OTel GenAI semantic conventions have no cost attribute. Unlike OpenInference (which has `llm.cost.total`), cost tracking is not part of the spec. Kalibra reports this as N/A, not $0.00. If your platform adds a vendor-specific cost attribute, you can map it via `fields.cost` in the config.

The `regressions <= 2` gate caught the real problem: 14 task types regressed when measured individually. The aggregate couldn't even confirm a direction — the breakdown was definitive.

## Your project config — `kalibra.yml`

This is the config that drives the comparison above. The notebook passes the same gates and fields programmatically, but in your project you'd use this file:

```yaml
sources:
  baseline:
    path: traces.jsonl
    where:
      - variant == baseline
  current:
    path: traces.jsonl
    where:
      - variant == current

baseline: baseline
current: current

fields:
  task_id: task.id

require:
  - token_delta_pct <= -10
  - regressions <= 2
```

**`where` clauses** split a single trace file into populations. Your filter field can be anything in the root span's attributes — `variant`, `git_sha`, `branch`, `experiment_id`.

**`fields.task_id`** tells the trace breakdown which attribute identifies distinct tasks. Without it, per-task regression detection is disabled.

**No `fields.cost`** is needed here — but if your tracing backend adds cost as a span attribute, map it:

```yaml
fields:
  cost: your_platform.cost_field
```

From your terminal: `kalibra compare --config kalibra.yml -v`

## Terminal output

This is what `kalibra compare --config kalibra.yml -v` produces — the same analysis, formatted for your terminal.

In [ ]:
print(render(result, "terminal", verbose=True))

In [ ]:
print(f"Direction: {result.direction.value}")
print(f"Gates passed: {result.passed}")
for gate in result.gates:
    if gate.warning and "skipped" in gate.warning.lower():
        status = "SKIP"
    elif gate.passed:
        status = "PASS"
    else:
        status = "FAIL"
    print(f"  [{status}] {gate.expr}  (actual: {gate.actual})")

if not result.passed:
    print("\nThis change would be blocked in CI.")
else:
    print("\nAll gates passed — change is within budget.")

## Exploring the trace data

OTel GenAI traces have a specific shape. Let's compare a successful baseline trace with a truncated current trace to see exactly which attributes drove the outcome detection.

In [ ]:
# ── Trace structure ───────────────────────────────────────────────────
t = baseline[0]
print(f"Example trace ({t.trace_id[:12]}...):")
print(f"  {len(t.spans)} spans, {t.total_tokens} tokens, outcome={t.outcome}")
for i, s in enumerate(t.spans):
    prefix = "  └──" if i == len(t.spans) - 1 else "  ├──"
    tokens = f"tokens={s.total_tokens}" if s.total_tokens else "no tokens"
    print(f"{prefix} {s.name} ({tokens})")

# ── Finish reasons from raw data ─────────────────────────────────────
# The loader consumes gen_ai.response.finish_reasons during outcome
# detection, so we peek at the raw JSONL to show the actual attribute.
print("\nFinish reasons (first truncated trace):")
with open(TRACES_PATH) as f:
    for line in f:
        span = json.loads(line)
        fr = span.get("attributes", {}).get("gen_ai.response.finish_reasons")
        if fr and fr != ["stop"]:
            print(f"  {span['name']}: gen_ai.response.finish_reasons = {fr}")
            print(f"  Kalibra maps this to: outcome = failure")
            break

# ── Side-by-side comparison ───────────────────────────────────────────
raw_spans = {}
with open(TRACES_PATH) as f:
    for line in f:
        span = json.loads(line)
        tid = span.get("context", {}).get("trace_id", "")
        raw_spans.setdefault(tid, []).append(span)

def show_trace(label, trace):
    print(f"--- {label} ---")
    print(f"  trace_id:     {trace.trace_id[:16]}...")
    print(f"  outcome:      {trace.outcome}")
    print(f"  total_tokens: {trace.total_tokens}")
    print(f"  total_cost:   {trace.total_cost}")
    print(f"  duration:     {trace.duration:.2f}s" if trace.duration else "  duration:     N/A")

    for raw in raw_spans.get(trace.trace_id, []):
        attrs = raw.get("attributes", {})
        if attrs.get("gen_ai.operation.name"):
            print(f"\n  LLM span: {raw['name']}")
            for k, v in sorted(attrs.items()):
                if k.startswith("gen_ai."):
                    print(f"    {k}: {v}")
            break
    print()

b_trace = next(t for t in baseline if t.outcome == "success"
               and t.metadata.get("task.id") == "architecture")
c_trace = next(t for t in current if t.outcome == "failure"
               and t.metadata.get("task.id") == "architecture")

show_trace("Baseline (success)", b_trace)
show_trace("Current (truncated)", c_trace)

print("Notice: total_cost is None on both — OTel GenAI has no cost attribute.")
print("Kalibra reports N/A, not $0.00.")

## Platform compatibility

Kalibra auto-detects `gen_ai.*` attributes. These traces came from `opentelemetry-instrumentation-openai-v2` through Phoenix. The same analysis should work with any exporter that preserves standard `gen_ai.*` span attributes:

| Platform | Expected support | Notes |
|----------|-----------------|-------|
| **`opentelemetry-instrumentation-openai-v2`** | Validated | What this notebook uses |
| **PydanticAI / Logfire** | Expected | Built on OTel |
| **Langfuse** | Expected | May include custom cost attributes — map via `fields.cost` |
| **Datadog LLM Observability** | Expected | Uses `gen_ai.*` conventions |
| **OpenLLMetry (Traceloop)** | Expected | OTel-based auto-instrumentation |

"Expected" means the platform uses OTel GenAI conventions based on their documentation, but Kalibra has not been tested against their specific export format. If you hit issues, [open an issue](https://github.com/khan5v/kalibra/issues).

## Why not just check success rate?

In this demo, success rate drops from 100% to 44% — obvious. You could catch this with a simple threshold. But this is a deliberately extreme scenario to make the mechanics visible.

In practice, regressions are subtler. One task type breaks while 24 others improve slightly. The aggregate success rate stays at 95% or even goes up. A token-level dashboard shows "costs are down." Nobody investigates.

The per-task breakdown catches the one that broke. That's what `regressions <= 2` does — it counts how many individual task types got worse, regardless of what the aggregate says. Kalibra then gives you the statistical confidence to distinguish real regressions from noise.

## Why aggregate metrics lie

Token usage dropped 62%. In any dashboard, the aggregate graph goes down. That looks like an improvement.

But Kalibra's statistical test reveals the confidence interval spans from -81% to +49% — the aggregate change isn't even statistically significant. And `gen_ai.response.finish_reasons` tells you why: complex tasks returned `["length"]` instead of `["stop"]`. The model did not finish its response. Tokens went down because outputs were cut short, not because the agent got more efficient. The high variance between simple tasks (unchanged) and truncated tasks (dramatically fewer tokens) is what blows the confidence interval wide open.

Meanwhile, the per-task breakdown is definitive: 14 task types went from 100% success to 0% success. No ambiguity, no wide confidence intervals. The `regressions <= 2` gate caught it.

See [Aggregate metrics are a blind spot in agent evaluation](https://orekhov.work/posts/span-tree-aggregation/) for harder cases where the aggregate does not move at all.

## Next steps

- **Point at your own traces.** Export from your OTel collector or tracing backend as JSONL. Run `kalibra inspect your_traces.jsonl` to check coverage, then `kalibra compare`.
- **Gate your CI.** `kalibra compare` returns exit 1 when any gate fails. Use [`khan5v/kalibra-action`](https://github.com/khan5v/kalibra-action) to gate PRs with a Markdown report.
- **Add cost tracking.** If your platform provides cost data as a span attribute, map it: `fields.cost: your_cost_field` in `kalibra.yml`.
- **Run more tasks.** More traces = tighter confidence intervals. Kalibra warns when sample sizes are below 30.

---

[Kalibra](https://github.com/khan5v/kalibra) · [Docs](https://kalibra.cc) · [GitHub Action](https://github.com/khan5v/kalibra-action) · [OTel GenAI Spec](https://opentelemetry.io/docs/specs/semconv/gen-ai/)

---

## Appendix: Generate your own traces

To reproduce this experiment with live API calls, you need an `OPENAI_API_KEY`. The code below instruments OpenAI with `opentelemetry-instrumentation-openai-v2`, runs the same 25 tasks with both `max_tokens` settings through Phoenix, and exports to JSONL.

In [ ]:
import os

# Colab: reads from Colab Secrets if available.
try:
    from google.colab import userdata
    os.environ["OPENAI_API_KEY"] = userdata.get("OPENAI_API_KEY")
except ImportError:
    pass

if not os.environ.get("OPENAI_API_KEY"):
    print("Skipped — no OPENAI_API_KEY set.")
    print("Set it to generate fresh traces, or use the pre-recorded ones above.")
else:
    import subprocess, sys, json, time
    subprocess.check_call([
        sys.executable, "-m", "pip", "install", "-q",
        "openai", "opentelemetry-instrumentation-openai-v2",
        "arize-phoenix", "arize-phoenix-otel",
    ])

    import phoenix as px
    from phoenix.otel import register
    from phoenix.client import Client
    from opentelemetry.instrumentation.openai_v2 import OpenAIInstrumentor
    from opentelemetry import trace as otel_trace
    from openai import OpenAI

    px.close_app()
    session = px.launch_app()
    tracer_provider = register()
    OpenAIInstrumentor().instrument(tracer_provider=tracer_provider)
    tracer = otel_trace.get_tracer("kalibra-otel-genai-demo")
    client = OpenAI()
    print(f"Phoenix UI: {session.url}")

    TASKS = [
        {"id": "translate",    "prompt": "Translate 'good morning' to French, German, and Japanese. Just the translations."},
        {"id": "acronym",      "prompt": "What does API stand for?"},
        {"id": "haiku",        "prompt": "Write a haiku about a server crash."},
        {"id": "oneliner",     "prompt": "Explain what a hash table is in one sentence."},
        {"id": "boolean",      "prompt": "Is a two-proportion z-test parametric or non-parametric? One word answer."},
        {"id": "convert",      "prompt": "Convert 72F to Celsius. Show just the number."},
        {"id": "count",        "prompt": "How many bits in a byte?"},
        {"id": "abbreviate",   "prompt": "What does CRUD stand for in database operations?"},
        {"id": "binary",       "prompt": "What is 255 in binary?"},
        {"id": "emoji",        "prompt": "Represent the CI/CD pipeline in emojis. One per stage."},
        {"id": "opposite",     "prompt": "What is the opposite of 'idempotent' in API design?"},
        {"id": "sla",          "prompt": "What does SLA stand for in cloud computing?"},
        {"id": "sql",          "prompt": "Write a SQL query to find customers who made a purchase every month of 2024. Use table orders(customer_id, order_date, amount). Include comments explaining each part."},
        {"id": "tradeoffs",    "prompt": "Compare microservices vs monolith. Give 3 scenarios where each wins, with concrete reasoning and examples."},
        {"id": "debug",        "prompt": "A Python web server returns 502 errors under load. Walk through a systematic debugging process: what to check first, what tools to use, common root causes, and how to fix each one."},
        {"id": "architecture", "prompt": "Design a system that processes 10,000 webhook events per second with at-least-once delivery. Cover: queue choice, consumer design, retry strategy, dead letter handling, and monitoring."},
        {"id": "migration",    "prompt": "Plan a zero-downtime migration from PostgreSQL to a new schema that renames 5 columns, splits one table into two, and adds a new index. Include rollback strategy."},
        {"id": "ztest",        "prompt": "Explain the two-proportion z-test: what it tests, assumptions, formula, and a worked example with actual numbers."},
        {"id": "cap",          "prompt": "Explain the CAP theorem, why it's commonly misunderstood, and what the PACELC extension adds. Give a real-world example for each trade-off."},
        {"id": "refactor",     "prompt": "Show how to refactor a 50-line Python function that mixes I/O, validation, and business logic into clean, testable components. Show before and after code."},
        {"id": "review",       "prompt": "Write a thorough code review for this function:\ndef process(data):\n  results = []\n  for item in data:\n    if item.get('status') == 'active' and item.get('score', 0) > 0.5:\n      results.append({'id': item['id'], 'value': item['score'] * 100})\n  return sorted(results, key=lambda x: -x['value'])[:10]"},
        {"id": "pipeline",     "prompt": "Design a data pipeline that ingests 1M CSV rows daily from 3 sources with different schemas, validates, deduplicates, transforms into a star schema, and loads into a data warehouse. Cover error handling, monitoring, and backfill strategy."},
        {"id": "security",     "prompt": "Explain the OWASP top 3 web vulnerabilities. For each: what it is, a concrete code example showing the vulnerability, how to exploit it, and the fix. Use Python/Flask examples."},
        {"id": "consensus",    "prompt": "Compare Raft and Paxos consensus algorithms. For each: how leader election works, how log replication works, what happens during network partitions, and when to choose one over the other."},
        {"id": "regression",   "prompt": "Describe 3 statistical methods for detecting regressions in A/B tests. For each: how it works, when to use it, when it fails, and a code sketch."},
    ]

    for variant, max_tokens in [("baseline", 2048), ("current", 150)]:
        print(f"\n{variant.upper()}: max_tokens={max_tokens}")
        for run_id in range(4):
            for task in TASKS:
                with tracer.start_as_current_span("agent", attributes={
                    "variant": variant, "task.id": task["id"], "task.run_id": run_id,
                }):
                    response = client.chat.completions.create(
                        model="gpt-4o-mini",
                        max_tokens=max_tokens,
                        messages=[{"role": "user", "content": task["prompt"]}],
                    )
                    fr = response.choices[0].finish_reason
                    tag = " [TRUNCATED]" if fr == "length" else ""
                    print(f"  [{task['id']}-run{run_id}] {fr}{tag}")

    # Export
    time.sleep(5)
    phoenix_client = Client()
    all_spans = phoenix_client.spans.get_spans(project_identifier="default")
    with open("traces_live.jsonl", "w") as f:
        for span in all_spans:
            f.write(json.dumps(span, default=str) + "\n")
    print(f"\nExported {len(all_spans)} spans -> traces_live.jsonl")
    print("Re-run the cells above with TRACES_PATH = 'traces_live.jsonl' to analyze your fresh traces.")